<a href="https://colab.research.google.com/github/hibbo67/ai-engineering-101/blob/main/day29_rag_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install chromadb google-genai
!pip install pyngrok

In [1]:
from google.colab import userdata
from google import genai
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings

# 1. FAST load: Grab keys 1-10 instantly
print("Fetching keys...")
api_keys = []

# This list comprehension does it all at once without sleeping
try:
    api_keys = [userdata.get(f"GEMINI_KEY_{i}") for i in range(1, 8) if userdata.get(f"GEMINI_KEY_{i}")]
except Exception as e:
    print(f"Error fetching secrets: {e}")

if not api_keys:
    raise ValueError("No GEMINI_KEYs found! Check your Colab secrets.")

print(f"✅ Loaded {len(api_keys)} keys instantly.")

# 2. Setup Embedding Client
client = genai.Client(api_key=api_keys[2])

# 3. Setup ChromaDB
class GeminiEmbeddingFunction(EmbeddingFunction):
    def __init__(self): pass
    def __call__(self, input: Documents) -> Embeddings:
        vectors = []
        for text in input:
            result = client.models.embed_content(
                model="gemini-embedding-2",
                contents=[text],
            )
            vectors.append([float(x) for x in result.embeddings[0].values])
        return vectors

db = chromadb.Client()
col = db.get_or_create_collection(name="support", embedding_function=GeminiEmbeddingFunction())

if col.count() == 0:
    docs = [
        "Our refund policy allows returns within 30 days.",
        "Shipping is free for orders over $50.",
        "Support hours: 9am–5pm EST, Mon–Fri.",
    ]
    col.add(ids=[f"d{i}" for i in range(len(docs))], documents=docs)
    print("Database populated.")

Fetching keys...
✅ Loaded 7 keys instantly.
Database populated.


In [2]:
from fastapi import FastAPI
from pydantic import BaseModel
import time
import asyncio
from google import genai
from google.genai import types # Required for HttpOptions

app = FastAPI()

class Question(BaseModel):
    q: str

def get_rag_answer(question):
    # 1. Retrieve
    hits = col.query(query_texts=[question], n_results=2)
    retrieved_docs = hits["documents"][0]
    distances = hits["distances"][0]

    # 2. Guardrail
    if distances[0] > 0.75:
        return "I don't know."

    # 3. Augment
    context = "\n".join(retrieved_docs)
    prompt = f"Answer only from context. If missing, say 'I don't know'.\n\nContext:\n{context}\n\nQ: {question}"

    # 4. Generate
    for i, key in enumerate(api_keys):
        key_num = i + 1
        try:
            print(f"  [API] Trying Key {key_num}...")

            # We add a 10-second timeout to the API call so it doesn't freeze Colab
            temp_client = genai.Client(
                api_key=key,
                http_options=types.HttpOptions(timeout=10_000) # 10,000 milliseconds
            )

            response = temp_client.models.generate_content(
                model="gemini-3.6-flash",
                config={"thinking_config": {"thinking_level": "low"}},
                contents=prompt
            )

            print(f"  [API] Success with Key {key_num}!")
            return response.text

        except Exception as e:
            error_msg = str(e)
            print(f"  [API] Key {key_num} failed: {error_msg[:50]}...")
            if "429" in error_msg or "503" in error_msg or "400" in error_msg or "DeadlineExceeded" in error_msg:
                time.sleep(1)
                continue
            else:
                raise e

    return "API limit reached on all keys."

@app.post("/ask")
async def ask(body: Question):
    # Run the blocking RAG function in a separate thread so the server stays alive
    answer = await asyncio.to_thread(get_rag_answer, body.q)
    return {"question": body.q, "answer": answer}

In [5]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn
import threading

nest_asyncio.apply()

# Kill any lingering tunnels
try:
    ngrok.kill()
except:
    pass

# Connect ngrok
public_url = ngrok.connect(8000)
print(f"🔥 Your RAG API is live at: {public_url}/ask")

# Configure Uvicorn
config = uvicorn.Config(app, port=8000, log_level="info")
server = uvicorn.Server(config)

# Start the server in a dedicated background thread
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()

🔥 Your RAG API is live at: NgrokTunnel: "https://woof-recharger-arrogant.ngrok-free.dev" -> "http://localhost:8000"/ask


INFO:     Started server process [8239]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


In [6]:
import requests

# The full URL to your API
url = "https://woof-recharger-arrogant.ngrok-free.dev/ask"
headers = {"ngrok-skip-browser-warning": "true"}

# Test 1: A question inside the database
print("Testing Refund Policy...")
res1 = requests.post(url, json={"q": "What is the refund policy?"}, headers=headers)
print(res1.json())

# Test 2: A question outside the database (should trigger the guardrail)
print("\nTesting Australia Shipping...")
res2 = requests.post(url, json={"q": "Do you ship to Australia?"}, headers=headers)
print(res2.json())

Testing Refund Policy...
  [API] Trying Key 1...
  [API] Success with Key 1!
INFO:     35.231.187.170:0 - "POST /ask HTTP/1.1" 200 OK
{'question': 'What is the refund policy?', 'answer': 'The refund policy allows returns within 30 days.'}

Testing Australia Shipping...
  [API] Trying Key 1...
  [API] Success with Key 1!
INFO:     35.231.187.170:0 - "POST /ask HTTP/1.1" 200 OK
{'question': 'Do you ship to Australia?', 'answer': "I don't know."}


In [7]:
import requests
import time

url = "https://woof-recharger-arrogant.ngrok-free.dev/ask"
headers = {"ngrok-skip-browser-warning": "true"}

questions = [
    "What is the refund policy?",
    "Do you ship to Australia?",
    "When is support open?",
    "Do you ship to Canada?",
    "Is shipping free?"
]

for q in questions:
    print(f"Asking: {q}")
    try:
        res = requests.post(url, json={"q": q}, headers=headers, timeout=15)
        print(f"  Answer: {res.json().get('answer', 'Error')}\n")
    except Exception as e:
        print(f"  Request failed: {e}\n")
    time.sleep(1) # Small pause between requests

Asking: What is the refund policy?
  [API] Trying Key 1...
  [API] Success with Key 1!
INFO:     35.231.187.170:0 - "POST /ask HTTP/1.1" 200 OK
  Answer: The refund policy allows returns within 30 days.

Asking: Do you ship to Australia?
  [API] Trying Key 1...
  [API] Success with Key 1!
INFO:     35.231.187.170:0 - "POST /ask HTTP/1.1" 200 OK
  Answer: I don't know.

Asking: When is support open?
  [API] Trying Key 1...
  [API] Success with Key 1!
INFO:     35.231.187.170:0 - "POST /ask HTTP/1.1" 200 OK
  Answer: Support is open from 9am–5pm EST, Monday through Friday.

Asking: Do you ship to Canada?
  [API] Trying Key 1...
  [API] Success with Key 1!
INFO:     35.231.187.170:0 - "POST /ask HTTP/1.1" 200 OK
  Answer: I don't know.

Asking: Is shipping free?
  [API] Trying Key 1...
  [API] Success with Key 1!
INFO:     35.231.187.170:0 - "POST /ask HTTP/1.1" 200 OK
  Answer: Shipping is free for orders over $50.

